# BrownDye2 association-rate preparation

BrownDye **stage only**. The APBS electrostatics stage lives in
`examples/apbs/{protein,ligand,complex}/` - each notebook there writes
`<name>.pqr`, `<name>.dx`, and `<name>.apbs.log` under its own `tmp/`.
This notebook reuses any two of those component outputs **as fixtures**
and builds the BrownDye simulation input; `bdrun.sh` then propagates the
trajectories.

Three-stage pipeline:

1. **APBS** (`examples/apbs/<name>/<name>_apbs.ipynb`) -> `<name>.pqr` + `<name>.dx`.
2. **BrownDye prep** (this notebook): pick two bodies, build
   `${CORE0}_${CORE1}_simulation.xml`.
3. **BrownDye run** (`bdrun.sh`): `nam_simulation` / `we_simulation` + rate constant.

Pick the two bodies with `CORE0` / `CORE1` in the first code cell (any of
`protein`, `ligand`, `complex`). The two bodies must be **co-registered in a
bound pose**; `protein` + `ligand` (the default) are split from the same
`complex.pdb`, so the ligand sits in its bound site relative to the protein -
a meaningful ligand-to-receptor association. (`complex` overlaps either single
body spatially, so other pairings are for plumbing demonstration only.)

Prerequisite: run the APBS notebooks for the two chosen components first, then
activate the AmberTools environment (BrownDye binaries are added to PATH below):

```bash
conda activate ambertools
```

In [ ]:
import os
from pathlib import Path

from mdpp.prep import (
    BrownDyeBody,
    BrownDyeSolvent,
    infer_debye_length,
    write_contact_types,
    write_input_xml,
)

# This directory; the APBS stage lives in the sibling examples/apbs/ tree.
EXAMPLE_DIR = Path.cwd()
APBS_EXAMPLES = EXAMPLE_DIR.parent / "apbs"

# --- Pick the two BrownDye bodies (APBS outputs reused as fixtures) ---
# Any of "protein", "ligand", "complex" (must be co-registered in a bound pose).
CORE0 = "protein"  # body 0 (held fixed; the receptor)
CORE1 = "ligand"  # body 1 (diffuses; the ligand)

# Per-component BrownDye metadata.
COMPONENT_IS_PROTEIN = {"protein": True, "ligand": False, "complex": True}
CORE0_DIELECTRIC = 4.0
CORE1_DIELECTRIC = 4.0


def apbs_outputs(name: str) -> tuple[Path, Path, Path]:
    """Resolve (pqr, dx, apbs_log) for an APBS component, or raise if missing."""
    base = APBS_EXAMPLES / name / "tmp"
    pqr = base / "ambertools" / f"{name}.pqr"
    dx = base / "apbs" / f"{name}.dx"
    log = base / "apbs" / f"{name}.apbs.log"
    missing = [str(p) for p in (pqr, dx, log) if not p.is_file()]
    if missing:
        raise FileNotFoundError(
            f"APBS outputs for {name!r} not found: {missing}. "
            f"Run examples/apbs/{name}/{name}_apbs.ipynb first."
        )
    return pqr, dx, log


CORE0_PQR, CORE0_DX, CORE0_LOG = apbs_outputs(CORE0)
CORE1_PQR, CORE1_DX, CORE1_LOG = apbs_outputs(CORE1)

# Workspace: the BrownDye stage owns tmp/bdprep (and tmp/bdrun via bdrun.sh).
TMP_ROOT = EXAMPLE_DIR / "tmp"
BDPREP_DIR = TMP_ROOT / "bdprep"
BDPREP_INTERMEDIATE = BDPREP_DIR / "intermediate"
for path in (BDPREP_DIR, BDPREP_INTERMEDIATE):
    path.mkdir(parents=True, exist_ok=True)

# BrownDye install + bodies.
BD_HOME = Path(os.environ.get("BD_HOME", "/apps/browndye2"))
BD_BIN = Path(os.environ.get("BD_BIN", str(BD_HOME / "bin")))
BD_AUX = Path(os.environ.get("BD_AUX", str(BD_HOME / "aux")))

# BrownDye reaction criteria + run parameters consumed by input.xml / bdrun.sh.
RXN_SEARCH_DISTANCE = 5.5
RXN_DISTANCE = 10.0
RXN_NEEDED = 3
N_THREADS = 1
SEED = 11111111
N_TRAJECTORIES = 100_000
N_TRAJECTORIES_PER_OUTPUT = 10
MAX_N_STEPS = 1_000_000
RESULTS_FILE = "results.xml"
TRAJECTORY_FILE = "trajectory"  # base name for trajectory*.xml / .index.xml dumps
N_STEPS_PER_OUTPUT = 10  # BD-step stride when recording trajectories (1 = every step)
DEBYE_LENGTH: float | None = None  # inferred from the APBS logs in Step 1

# Export shell-facing knobs so the %%bash cells inherit the configuration.
os.environ.update({
    "TMP_ROOT": str(TMP_ROOT),
    "BDPREP_DIR": str(BDPREP_DIR),
    "BDPREP_INTERMEDIATE": str(BDPREP_INTERMEDIATE),
    "CORE0": CORE0,
    "CORE1": CORE1,
    "RXN_SEARCH_DISTANCE": f"{RXN_SEARCH_DISTANCE}",
    "RXN_DISTANCE": f"{RXN_DISTANCE}",
    "RXN_NEEDED": f"{RXN_NEEDED}",
    "BD_HOME": str(BD_HOME),
    "BD_BIN": str(BD_BIN),
    "BD_AUX": str(BD_AUX),
})
os.environ["PATH"] = f"{BD_BIN}:{BD_AUX}:{os.environ.get('PATH', '')}"
print(
    f"Bodies: CORE0={CORE0} (is_protein={COMPONENT_IS_PROTEIN[CORE0]}), "
    f"CORE1={CORE1} (is_protein={COMPONENT_IS_PROTEIN[CORE1]})"
)

## Step 1: Link APBS fixtures and infer the Debye length

Symlink each chosen body's `.pqr` (charges + radii) and `.dx` (potential grid)
from `examples/apbs/<name>/tmp/` into this stage's prep folder, then parse the
Debye length from the two APBS logs (BrownDye needs a finite Debye length for
the far-field electrostatics).

The fixtures are **symlinks, not copies**, so the BrownDye inputs always track
the latest APBS run. The pipeline becomes: change a component's PDB -> re-run
its `examples/apbs/<name>/<name>_apbs.ipynb` -> re-run this notebook, with no
manual copying in between.

In [ ]:
# Symlink each body's APBS outputs into the BrownDye prep folder. Links (not
# copies) mean the inputs always track the latest APBS run: change a PDB, re-run
# its examples/apbs/<name>/<name>_apbs.ipynb, then just re-run this notebook.
def link_fixture(src: Path, dst: Path) -> None:
    """Point dst at src via a symlink, replacing any existing file or link."""
    if dst.is_symlink() or dst.exists():
        dst.unlink()
    dst.symlink_to(src)


link_fixture(CORE0_PQR, BDPREP_INTERMEDIATE / f"{CORE0}.pqr")
link_fixture(CORE1_PQR, BDPREP_INTERMEDIATE / f"{CORE1}.pqr")
link_fixture(CORE0_DX, BDPREP_INTERMEDIATE / f"{CORE0}.dx")
link_fixture(CORE1_DX, BDPREP_INTERMEDIATE / f"{CORE1}.dx")

# Infer the Debye length from the APBS logs unless pinned manually above.
if DEBYE_LENGTH is None:
    DEBYE_LENGTH = infer_debye_length(CORE0_LOG, CORE1_LOG)
os.environ["DEBYE_LENGTH"] = f"{DEBYE_LENGTH}"
print(f"Debye length: {DEBYE_LENGTH} A")

## Step 2: PQR -> BrownDye atoms.xml

`pqr2xml` converts each staged PQR into a BrownDye `atoms.xml`.

In [ ]:
%%bash
set -euo pipefail
cd "$BDPREP_INTERMEDIATE"

echo "=== PQR -> BrownDye XML ==="
pqr2xml <"${CORE0}.pqr" >"${CORE0}_atoms.xml"
pqr2xml <"${CORE1}.pqr" >"${CORE1}_atoms.xml"

## Step 3: Contact types

`write_contact_types` lists every unique heavy-atom `(atom_name, residue_name)`
per body; `make_rxn_pairs` consumes it to enumerate candidate contact pairs.

In [ ]:
contact_types_path = write_contact_types(
    BDPREP_INTERMEDIATE / f"{CORE0}.pqr",
    BDPREP_INTERMEDIATE / f"{CORE1}.pqr",
    BDPREP_INTERMEDIATE / "contact_types.xml",
)
print("contact_types.xml ->", contact_types_path)

## Step 4: Reaction criteria

`make_rxn_pairs` finds bound-pose contact pairs within `RXN_SEARCH_DISTANCE`;
`make_rxn_file` turns them into a reaction that fires when `RXN_NEEDED` pairs
come within `RXN_DISTANCE`. This is where the two bodies must be co-registered
in a bound pose - otherwise no contact pairs are found.

In [ ]:
%%bash
set -euo pipefail
cd "$BDPREP_INTERMEDIATE"

echo "=== Reaction criteria ==="
make_rxn_pairs -nonred \
    -mol0 "${CORE0}_atoms.xml" \
    -mol1 "${CORE1}_atoms.xml" \
    -ctypes contact_types.xml \
    -dist "$RXN_SEARCH_DISTANCE" >reaction_pairs.xml
make_rxn_file \
    -pairs reaction_pairs.xml \
    -state_from before \
    -state_to after \
    -rxn association \
    -mol0 "$CORE0" "$CORE0" \
    -mol1 "$CORE1" "$CORE1" \
    -distance "$RXN_DISTANCE" \
    -nneeded "$RXN_NEEDED" >reactions.xml

n_pairs=$(grep -c "<pair>" reaction_pairs.xml || true)
echo "contact pairs found: $n_pairs (need >= $RXN_NEEDED for association)"

## Step 5: BrownDye top-level input.xml

`write_input_xml` assembles the two `BrownDyeBody` descriptors (atoms.xml +
potential grid + `is_protein` + dielectric) and the shared `BrownDyeSolvent`
(Debye length) into the top-level `input.xml` consumed by `bd_top`.

In [ ]:
input_xml = write_input_xml(
    BDPREP_INTERMEDIATE / "input.xml",
    BrownDyeBody(
        name=CORE0,
        atoms_xml=f"{CORE0}_atoms.xml",
        grid_dx=f"{CORE0}.dx",
        is_protein=COMPONENT_IS_PROTEIN[CORE0],
        dielectric=CORE0_DIELECTRIC,
    ),
    BrownDyeBody(
        name=CORE1,
        atoms_xml=f"{CORE1}_atoms.xml",
        grid_dx=f"{CORE1}.dx",
        is_protein=COMPONENT_IS_PROTEIN[CORE1],
        dielectric=CORE1_DIELECTRIC,
    ),
    solvent=BrownDyeSolvent(debye_length_a=DEBYE_LENGTH),
    n_threads=N_THREADS,
    seed=SEED,
    n_trajectories=N_TRAJECTORIES,
    n_trajectories_per_output=N_TRAJECTORIES_PER_OUTPUT,
    max_n_steps=MAX_N_STEPS,
    n_steps_per_output=N_STEPS_PER_OUTPUT,
    results_file=RESULTS_FILE,
    trajectory_file=TRAJECTORY_FILE,
)
print("input.xml ->", input_xml)

## Step 6: bd_top -> simulation.xml

`bd_top` validates `input.xml` and writes `${CORE0}_${CORE1}_simulation.xml`,
the file `bdrun.sh` consumes. The prep artifacts are copied up to
`tmp/bdprep/` for inspection.

In [ ]:
%%bash
set -euo pipefail
cd "$BDPREP_INTERMEDIATE"

echo "=== BrownDye top-level input ==="
bd_top input.xml

for file in \
    "${CORE0}_atoms.xml" \
    "${CORE1}_atoms.xml" \
    contact_types.xml \
    reaction_pairs.xml \
    reactions.xml \
    input.xml \
    "${CORE0}_${CORE1}_simulation.xml"; do
    cp "$file" "$BDPREP_DIR/$file"
done

echo "Simulation XML: $BDPREP_DIR/${CORE0}_${CORE1}_simulation.xml"

## Step 7: Run BrownDye trajectories

Trajectory propagation (`N_TRAJECTORIES=100_000`) can take hours, so it stays a
shell script. From a terminal in this directory, pass the same body names so
`bdrun.sh` reads the matching simulation XML:

```bash
conda activate ambertools
cd examples/browndye
CORE0=protein CORE1=ligand bash bdrun.sh            # standard NAM mode
# or
CORE0=protein CORE1=ligand MODE=we bash bdrun.sh    # weighted-ensemble mode
```

`bdrun.sh` reads `tmp/bdprep/intermediate/${CORE0}_${CORE1}_simulation.xml`
(written by Step 6) and writes `tmp/bdrun/results.xml` plus
`tmp/bdrun/rate_constant.txt`.

## Step 8: Visualize a reactive trajectory

When `<trajectory_file>` and `<n_steps_per_output>` are present in `input.xml`
(Step 5), `nam_simulation` writes `trajectoryN.xml` plus `trajectoryN.index.xml`
(one pair per thread) into `tmp/bdrun/intermediate/`. To turn one reactive
trajectory into a VTF animation for VMD:

1. `process_trajectories -srxn association` lists the reactive trajectory indices.
2. `process_trajectories -n <idx> -nstride <stride>` decompresses one trajectory.
3. `vtf_trajectory` converts it to `trajectory.vtf` (`vmd trajectory.vtf`).

Tune via env vars before running the cell: `TRAJ_THREAD` (default `0`),
`TRAJ_IDX` (default: first reactive trajectory), `STRIDE` (default `10`).

In [ ]:
%%bash
set -euo pipefail
BDRUN_DIR="${BDRUN_DIR:-$TMP_ROOT/bdrun}"
BDRUN_INTERMEDIATE="${BDRUN_INTERMEDIATE:-$BDRUN_DIR/intermediate}"
TRAJ_THREAD="${TRAJ_THREAD:-0}"
STRIDE="${STRIDE:-10}"

traj_xml="trajectory${TRAJ_THREAD}.xml"
traj_index="trajectory${TRAJ_THREAD}.index.xml"

if [[ ! -s "$BDRUN_INTERMEDIATE/$traj_xml" || ! -s "$BDRUN_INTERMEDIATE/$traj_index" ]]; then
    echo "No BrownDye trajectory dumps in $BDRUN_INTERMEDIATE yet."
    echo "Run 'CORE0=$CORE0 CORE1=$CORE1 bash bdrun.sh' first, then re-run this cell."
    exit 0
fi

cd "$BDRUN_INTERMEDIATE"
echo "=== Reactive trajectories in $traj_xml ==="
process_trajectories \
    -traj "$traj_xml" \
    -index "$traj_index" \
    -srxn association | tee reactive_trajectories.txt

# Default to the first reactive trajectory number unless TRAJ_IDX is set.
TRAJ_IDX="${TRAJ_IDX:-$(grep -oE '[0-9]+' reactive_trajectories.txt | head -n1 || true)}"
if [[ -z "$TRAJ_IDX" ]]; then
    echo "No reactive trajectories found in $traj_xml" >&2
    exit 1
fi
echo "Selected reactive trajectory index=$TRAJ_IDX, stride=$STRIDE"

echo "=== Decompress to trajectory.xml ==="
process_trajectories \
    -traj "$traj_xml" \
    -index "$traj_index" \
    -n "$TRAJ_IDX" \
    -nstride "$STRIDE" >trajectory.xml

echo "=== Estimate VTF size ==="
vtf_trajectory \
    -mol0 "${CORE0}_atoms.xml" \
    -mol1 "${CORE1}_atoms.xml" \
    -trajf trajectory.xml \
    -trial || true

echo "=== Generate trajectory.vtf for VMD ==="
vtf_trajectory -traj trajectory.xml >trajectory.vtf

mkdir -p "$BDRUN_DIR"
cp reactive_trajectories.txt "$BDRUN_DIR/reactive_trajectories.txt"
cp trajectory.xml "$BDRUN_DIR/trajectory.xml"
cp trajectory.vtf "$BDRUN_DIR/trajectory.vtf"
echo
ls -lh "$BDRUN_DIR/trajectory.vtf"
echo "Load in VMD with:  vmd $BDRUN_DIR/trajectory.vtf"